# PCA y Kernel PCA — Ejemplos

**Módulo 3 — Modelado y evaluación de enfoques no supervisados · Curso Analítica de Datos**

Este notebook acompaña las diapositivas [`3.5_Que_es_Principal_Component_Analysis_PCA.pdf`](3.5_Que_es_Principal_Component_Analysis_PCA.pdf) y lleva a código lo que allí se explica.

## Contenido

1. **PCA paso a paso, desde cero**, con un dataset de juguete: centrar los datos, aplicar descomposición en valores singulares (SVD), ordenar y elegir componentes, y proyectar — verificando al final que el resultado coincide con `scikit-learn`.
2. **PCA como función** (`sklearn.decomposition.PCA`) aplicado a un dataset real: la composición química de vinos (*Wine dataset*).
3. **Kernel PCA**: por qué una proyección lineal no siempre alcanza, y cómo el *kernel trick* permite separar estructuras no lineales que PCA no puede.

> 💡 Este notebook **no requiere descargar nada ni tener cuenta de Kaggle**: el dataset de juguete se genera en el propio notebook y el *Wine dataset* viene incluido directamente en `scikit-learn`. Solo necesitas tener instalado `scikit-learn`, además de `pandas`, `numpy` y `matplotlib`.

In [ ]:
# Librerías que usaremos en todo el notebook
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine, make_circles
from sklearn.decomposition import PCA, KernelPCA
from sklearn.preprocessing import StandardScaler

pd.set_option('display.max_columns', 20)
plt.rcParams['figure.figsize'] = (7, 5)
rng = np.random.default_rng(42)  # semilla fija para que los ejemplos sean reproducibles

print('Librerías cargadas correctamente ✅')
print('scikit-learn listo para usarse')

---
## 1. PCA paso a paso, desde cero

### 1.1 Un dataset de juguete

Simulamos las calificaciones de 200 estudiantes ficticios en tres materias — Matemáticas, Física y Química — a propósito **correlacionadas entre sí** (quien es fuerte en Matemáticas tiende a ser fuerte también en Física y Química, con algo de variación individual). Es un ejemplo pequeño e intuitivo: si existe un "desempeño académico general", debería aparecer como la dirección de mayor variabilidad de los datos.

In [ ]:
n = 200
matematicas = rng.normal(70, 12, n)
fisica = 0.85 * matematicas + rng.normal(0, 6, n)
quimica = 0.75 * matematicas + rng.normal(0, 8, n)

datos = pd.DataFrame({'Matematicas': matematicas, 'Fisica': fisica, 'Quimica': quimica})
print(f'{datos.shape[0]} estudiantes, {datos.shape[1]} materias')
print('\nCorrelaciones entre materias:')
print(datos.corr().round(2))
datos.head()

### 1.2 Intuición geométrica: la dirección de máxima varianza

Antes de trabajar con las tres materias a la vez, veamos el caso más simple posible con solo dos (Matemáticas y Física), como en las diapositivas 2 a 4: entre todas las direcciones posibles, ¿cuál conserva más información al proyectar los puntos sobre ella?

In [ ]:
X2 = datos[['Matematicas', 'Fisica']].values
X2_centrada = X2 - X2.mean(axis=0)

# SVD sobre estas dos variables: la primera fila de Vt (V transpuesta) es la
# dirección de máxima varianza (PC1); la segunda es la dirección ortogonal restante (PC2).
_, S2, Vt2 = np.linalg.svd(X2_centrada, full_matrices=False)
pc1_2d, pc2_2d = Vt2[0], Vt2[1]
centro_2d = X2.mean(axis=0)
escala = S2[0] / np.sqrt(n - 1) * 2.5  # solo para que la flecha se vea proporcional a la nube de puntos

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(X2[:, 0], X2[:, 1], alpha=0.4, s=15)
ax.annotate('', xy=centro_2d + pc1_2d * escala, xytext=centro_2d - pc1_2d * escala,
            arrowprops=dict(arrowstyle='->', color='green', lw=2))
ax.annotate('', xy=centro_2d + pc2_2d * escala * 0.4, xytext=centro_2d - pc2_2d * escala * 0.4,
            arrowprops=dict(arrowstyle='->', color='orange', lw=2))
ax.set_xlabel('Matemáticas'); ax.set_ylabel('Física')
ax.set_title('PC1 (verde, máxima varianza) y PC2 (naranja, varianza restante)')
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

print('La flecha verde (PC1) apunta en la dirección donde los puntos están más dispersos:')
print('la diagonal que sigue la nube de datos. La flecha naranja (PC2), perpendicular a la')
print('primera, apunta hacia donde queda la varianza restante — mucho menor.')

### 1.3 Los cinco pasos de PCA (con las tres materias)

Ahora sí, con las tres variables a la vez, seguimos exactamente los cinco pasos de la diapositiva 5: organizar la matriz, centrar, aplicar SVD, ordenar por varianza y quedarnos con los primeros componentes.

In [ ]:
# Paso 1: organizar los datos en una matriz X (n observaciones x p variables)
X = datos.values
print(f'Paso 1 — Matriz X: {X.shape[0]} filas x {X.shape[1]} columnas')

# Paso 2: centrar respecto a la media (¡fundamental! PCA es sensible al origen de coordenadas)
X_centrada = X - X.mean(axis=0)
print('Paso 2 — Media de cada columna tras centrar (debe dar ~0):', np.round(X_centrada.mean(axis=0), 8))

In [ ]:
# Paso 3: aplicar SVD -> X_centrada = U · Sigma · V^T
U, S, Vt = np.linalg.svd(X_centrada, full_matrices=False)
print('Paso 3 — SVD:')
print(f'  U (vectores singulares izquierdos): {U.shape}')
print(f'  S (valores singulares, en la diagonal de Sigma): {S.shape} -> {np.round(S, 2)}')
print(f'  Vt (V transpuesta; sus filas son las direcciones de los componentes principales): {Vt.shape}')

In [ ]:
# Paso 4: ordenar componentes por varianza explicada
# (numpy.linalg.svd ya entrega S de mayor a menor, así que solo lo confirmamos y lo calculamos)
varianza_explicada = (S ** 2) / (n - 1)          # varianza de cada componente
razon_varianza = varianza_explicada / varianza_explicada.sum()   # Explained Variance Ratio (diapositiva 7)
razon_acumulada = np.cumsum(razon_varianza)

tabla_varianza = pd.DataFrame({
    'componente': ['PC1', 'PC2', 'PC3'],
    'varianza_explicada_%': (razon_varianza * 100).round(2),
    'acumulada_%': (razon_acumulada * 100).round(2),
})
print('Paso 4 — Varianza explicada por componente:')
tabla_varianza

In [ ]:
fig, ax = plt.subplots()
ax.bar(range(1, 4), razon_varianza * 100, color='#2e7d32', label='Varianza individual (%)')
ax.plot(range(1, 4), razon_acumulada * 100, marker='o', color='#1565c0', label='Varianza acumulada (%)')
ax.axhline(95, linestyle='--', color='gray', label='Umbral 95%')
ax.set_xticks([1, 2, 3]); ax.set_xticklabels(['PC1', 'PC2', 'PC3'])
ax.set_ylabel('% de varianza')
ax.legend()
plt.tight_layout()
plt.show()

# ¿Cuántos componentes hacen falta para llegar al 95% (diapositiva 8)?
d_95 = int(np.searchsorted(razon_acumulada, 0.95) + 1)
print(f'Se necesitan {d_95} de los 3 componentes para alcanzar el 95% de varianza acumulada.')
print('Para VISUALIZAR, sin embargo, casi siempre basta con 2 componentes (diapositiva 8),')
print(f'aunque en este caso 2 componentes ya cubren un {razon_acumulada[1]:.1%} — muy cerca del umbral.')

In [ ]:
# Paso 5: proyectar sobre los primeros d=2 componentes -> X_d-proy = X_centrada @ W_d
W2 = Vt[:2].T  # las primeras 2 filas de Vt, como columnas: la matriz W_d de la diapositiva 6
X_proyectada_2d = X_centrada @ W2

print(f'Paso 5 — Proyección: de {X.shape[1]} variables originales a {X_proyectada_2d.shape[1]} componentes.')

fig, ax = plt.subplots()
ax.scatter(X_proyectada_2d[:, 0], X_proyectada_2d[:, 1], alpha=0.5, color='#6a1b9a')
ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
ax.set_title('Las tres materias, resumidas en 2 componentes principales')
plt.tight_layout()
plt.show()

print('Cada punto sigue siendo un estudiante, pero ahora descrito con solo 2 números (PC1, PC2)')
print('en vez de 3 calificaciones — combinaciones lineales de las materias originales.')

### 1.4 Verificación cruzada con `scikit-learn`

Confirmamos que nuestra implementación manual, paso a paso, coincide con lo que calcula `sklearn.decomposition.PCA`.

In [ ]:
pca_verificacion = PCA(n_components=3)
proyeccion_sklearn = pca_verificacion.fit_transform(X)  # PCA de sklearn centra internamente, no hace falta pasarle X_centrada

print('Razón de varianza explicada:')
print('  Cálculo manual: ', np.round(razon_varianza, 4))
print('  scikit-learn:   ', np.round(pca_verificacion.explained_variance_ratio_, 4))

# El signo de cada componente es arbitrario (V y -V son igual de válidos), así que
# comparamos con una correlación en vez de una igualdad directa.
for i in range(2):
    correlacion = np.corrcoef(X_proyectada_2d[:, i], proyeccion_sklearn[:, i])[0, 1]
    print(f'  Correlación PC{i+1} (manual vs. sklearn): {correlacion:.4f}  (±1.0 = coinciden, solo puede cambiar el signo)')

---
## 2. PCA como función: aplicación al dataset *Wine*

Usamos ahora `sklearn.decomposition.PCA` directamente sobre un dataset real: la composición química de 178 vinos de tres cultivares distintos (*Wine dataset*, incluido en `scikit-learn`).

In [ ]:
vino = load_wine(as_frame=True)
X_vino = vino.data
y_vino = vino.target

print(f'{X_vino.shape[0]} vinos, {X_vino.shape[1]} variables químicas, {len(vino.target_names)} cultivares: {list(vino.target_names)}')
X_vino.head()

⚠️ **Antes de aplicar PCA, hay que estandarizar.** PCA busca las direcciones de máxima varianza, así que una variable medida en una escala mucho más grande que las demás (por ejemplo `proline`, en cientos, frente a variables que van de 0 a 5) dominaría los componentes solo por su escala — no porque sea más informativa.

In [ ]:
comparacion_escalas = X_vino.agg(['mean', 'std']).T.round(2)
comparacion_escalas.loc[['proline', 'hue', 'nonflavanoid_phenols']]

In [ ]:
escalador = StandardScaler()
X_vino_estandarizada = escalador.fit_transform(X_vino)

# PCA con todos los componentes posibles, solo para ver cuánta varianza explica cada uno
pca_completo = PCA()
pca_completo.fit(X_vino_estandarizada)

razon_vino = pca_completo.explained_variance_ratio_
acumulada_vino = np.cumsum(razon_vino)
d95_vino = int(np.searchsorted(acumulada_vino, 0.95) + 1)

fig, ax = plt.subplots()
ax.bar(range(1, len(razon_vino) + 1), razon_vino * 100, color='#2e7d32')
ax.plot(range(1, len(razon_vino) + 1), acumulada_vino * 100, marker='o', color='#1565c0')
ax.axhline(95, linestyle='--', color='gray')
ax.set_xlabel('Componente'); ax.set_ylabel('% de varianza')
plt.tight_layout()
plt.show()

print(f'Se necesitan {d95_vino} de los {X_vino.shape[1]} componentes originales para llegar al 95% de varianza.')
print('A diferencia del dataset de juguete, aquí la información está más repartida entre variables:')
print('la composición química de un vino no se resume tan fácilmente en uno o dos números.')

Para **reducir dimensionalidad** en serio harían falta bastantes componentes, pero para **visualizar** (diapositiva 8) casi siempre usamos solo 2, aunque no lleguen al 95%:

In [ ]:
pca_2d = PCA(n_components=2)
X_vino_2d = pca_2d.fit_transform(X_vino_estandarizada)

print(f'Con solo 2 componentes se conserva el {pca_2d.explained_variance_ratio_.sum():.1%} de la varianza total.')

fig, ax = plt.subplots()
for clase, nombre in enumerate(vino.target_names):
    mascara = y_vino == clase
    ax.scatter(X_vino_2d[mascara, 0], X_vino_2d[mascara, 1], label=nombre, alpha=0.7)
ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
ax.set_title('Vinos proyectados a 2 componentes principales')
ax.legend()
plt.tight_layout()
plt.show()

print('Aunque PCA NUNCA vio la etiqueta del cultivar (es no supervisado), los tres grupos')
print('quedan claramente separados — la composición química por sí sola ya distingue los cultivares.')

In [ ]:
# ¿Qué variables originales pesan más en PC1? (los componentes son combinaciones lineales, diapositiva 6)
cargas = pd.DataFrame(pca_2d.components_.T, index=X_vino.columns, columns=['PC1', 'PC2'])
print('Las 3 variables químicas con mayor peso (en valor absoluto) en PC1:')
cargas['PC1'].abs().sort_values(ascending=False).head(3)

---
## 3. Kernel PCA: cuando una proyección lineal no alcanza

Como advierte la diapositiva 9, PCA solo encuentra subespacios **lineales**. El *Swiss Roll* del notebook `3.4` es un ejemplo real de esa limitación. Aquí usamos un caso más sencillo de visualizar — dos círculos concéntricos — para comparar de frente qué tanto logra separar cada método.

In [ ]:
X_circulos, y_circulos = make_circles(n_samples=400, factor=0.3, noise=0.05, random_state=0)

fig, ax = plt.subplots(figsize=(5, 5))
for clase, color in zip([0, 1], ['#1565c0', '#ef6c00']):
    mascara = y_circulos == clase
    ax.scatter(X_circulos[mascara, 0], X_circulos[mascara, 1], color=color, s=15)
ax.set_aspect('equal')
ax.set_title('Datos originales: un círculo dentro de otro')
plt.tight_layout()
plt.show()

print('Las dos clases NO son separables con una línea recta: cualquier recta que tracemos')
print('deja puntos de ambas clases a los dos lados.')

In [ ]:
pca_lineal = PCA(n_components=2)
X_circulos_pca = pca_lineal.fit_transform(X_circulos)

kernel_pca_rbf = KernelPCA(n_components=2, kernel='rbf', gamma=10)
X_circulos_kpca = kernel_pca_rbf.fit_transform(X_circulos)

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for clase, color in zip([0, 1], ['#1565c0', '#ef6c00']):
    mascara = y_circulos == clase
    axes[0].scatter(X_circulos_pca[mascara, 0], X_circulos_pca[mascara, 1], color=color, s=15)
    axes[1].scatter(X_circulos_kpca[mascara, 0], X_circulos_kpca[mascara, 1], color=color, s=15)

axes[0].set_title('PCA lineal — sigue mezclado')
axes[1].set_title('Kernel PCA (RBF) — ahora sí se separa')
plt.tight_layout()
plt.show()

def separabilidad(x, y):
    """Qué tan lejos quedan, en promedio, los centros de las dos clases en esta dimensión
    (en unidades de desviación estándar) — un número simple para comparar qué tanto separó cada método."""
    return abs(x[y == 0].mean() - x[y == 1].mean()) / (x.std() + 1e-9)

print(f'Separación en la primera dimensión con PCA lineal:  {separabilidad(X_circulos_pca[:, 0], y_circulos):.3f}')
print(f'Separación en la primera dimensión con Kernel PCA:  {separabilidad(X_circulos_kpca[:, 0], y_circulos):.3f}')
print('\nPCA lineal apenas rota/refleja los datos (la estructura circular no cambia).')
print('Kernel PCA, al mapear implícitamente a un espacio de mayor dimensión antes de aplicar PCA')
print('(el "kernel trick" de la diapositiva 10), logra que el círculo interior quede completamente')
print('separado del exterior.')

### La elección del kernel importa

La diapositiva 10 menciona tres kernels habituales (RBF/Gaussiano, Polinómico y Sigmoid) y advierte que la elección del kernel y sus hiperparámetros **afecta significativamente** la representación obtenida. Lo comprobamos:

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, kernel in zip(axes, ['rbf', 'poly', 'sigmoid']):
    parametros = dict(n_components=2, kernel=kernel)
    if kernel != 'poly':
        parametros['gamma'] = 10
    else:
        parametros['degree'] = 3
    kpca = KernelPCA(**parametros)
    proyeccion = kpca.fit_transform(X_circulos)

    for clase, color in zip([0, 1], ['#1565c0', '#ef6c00']):
        mascara = y_circulos == clase
        ax.scatter(proyeccion[mascara, 0], proyeccion[mascara, 1], color=color, s=15)
    ax.set_title(f"kernel = '{kernel}'")

plt.tight_layout()
plt.show()

print('Con estos mismos datos e hiperparámetros, el kernel RBF separa las clases con claridad,')
print('mientras que polinómico y sigmoid apenas modifican la estructura circular original.')
print('Elegir el kernel (y su(s) hiperparámetro(s), como gamma) no es un detalle menor: define')
print('si Kernel PCA realmente encuentra la estructura no lineal de los datos o no.')

---
## 4. Para pensar y discutir en clase

1. En el dataset de juguete (sección 1), PC1 explica cerca del 79% de la varianza. ¿Qué crees que representa esa dirección en términos "del mundo real" (pista: revisa qué tan parecidos son los tres pesos de `Matemáticas`, `Física` y `Química` en la primera fila de `Vt`)?
2. En el dataset *Wine* (sección 2), ¿por qué tiene sentido que se necesiten muchos más componentes para llegar al 95% de varianza que en el dataset de juguete? ¿Qué dice eso sobre qué tan "compresible" es cada dataset?
3. En la sección 3, ¿en qué otro tipo de datos esperarías que un kernel RBF ayude, y en cuáles crees que un kernel polinómico sería más apropiado? Justifica con la forma de los datos, no solo con el resultado que viste aquí.

No hay una única respuesta "correcta": lo importante es justificar cada respuesta con evidencia del propio notebook.

---
## Cierre

En este notebook llevamos a código las ideas de las diapositivas [`3.5_Que_es_Principal_Component_Analysis_PCA.pdf`](3.5_Que_es_Principal_Component_Analysis_PCA.pdf):

- Cómo implementar **PCA desde cero** con SVD: centrar, descomponer, ordenar por varianza y proyectar — y confirmar que coincide con `scikit-learn`.
- Cómo interpretar la **varianza explicada** (individual y acumulada) para decidir cuántos componentes conservar.
- Que PCA **siempre debe aplicarse después de estandarizar** cuando las variables tienen escalas muy distintas.
- Que PCA puede revelar estructura incluso sin usar etiquetas (los cultivares de vino se separaron sin que el algoritmo los viera).
- Por qué una proyección **lineal** no basta para estructuras no lineales, y cómo **Kernel PCA** resuelve eso mediante el *kernel trick* — y que la elección del kernel no es un detalle menor.

### Recursos adicionales
- [Documentación de `sklearn.decomposition.PCA`](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html)
- [Documentación de `sklearn.decomposition.KernelPCA`](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.KernelPCA.html)
- [Documentación de `numpy.linalg.svd`](https://numpy.org/doc/stable/reference/generated/numpy.linalg.svd.html)
- [Galería de ejemplos de reducción de dimensión de scikit-learn](https://scikit-learn.org/stable/auto_examples/decomposition/plot_kernel_pca.html)